# 🔬 Power Prediction ML Model — Research Paper (v2 — Complete)
### All 4 Power Types: Total Avg | Static | Dynamic (Worst Case) | Average Dynamic

**Gates:** Inverter, 2 i/p NAND, 2 i/p NOR  
**Temperature range:** -40°C to 120°C  
**Units:** pW (picowatts)

---
## 📋 Power Types in Your Dataset
| # | Power Type | Description |
|---|---|---|
| 1 | **Total Avg Power** | Overall average power consumed |
| 2 | **Static Power (worst case)** | Leakage/standby power — exponential with temp |
| 3 | **Dynamic Power (worst case)** | Switching power — worst case scenario |
| 4 | **Average Dynamic Power** | Average switching power across input combinations |

## 📦 Step 0: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
import joblib, os

print('✅ All libraries imported!')

## 📊 Step 1: Load ALL Power Data (4 Power Types × 3 Gates)

In [ ]:
# ================================================================
# YOUR COMPLETE RESEARCH DATA — ALL 4 POWER TYPES
# ================================================================

temps = [-40, -20, 0, 20, 27, 40, 60, 80, 100, 120]

# --- 1. TOTAL AVERAGE POWER (pW) ---
total_avg = {
    'Inverter': [1712,  1750,  1790,  1835,  1849,  1882,  1927,  1974,  2024,  2075],
    'NAND':     [7525,  7720,  7921,  8109,  8183,  8309,  8502,  8696,  8888,  9084],
    'NOR':      [8214,  8510,  8800,  9098,  9209,  9403,  9686,  9997, 10290, 10580],
}

# --- 2. STATIC POWER — WORST CASE (pW) ---
static_power = {
    'Inverter': [3.847, 3.979, 4.297, 4.983, 5.359,  6.318,  8.702, 12.67, 18.89, 28.19],
    'NAND':     [6.496, 6.527, 6.589, 6.805, 6.948,  8.081, 11.970, 18.35, 30.95, 53.78],
    'NOR':      [13.57, 13.85, 14.50, 15.89, 16.65, 18.570, 23.360, 31.30, 43.76, 62.38],
}

# --- 3. DYNAMIC POWER — WORST CASE (pW) ---
dynamic_worst = {
    'Inverter': [1708.153, 1746.021, 1785.703, 1830.017, 1843.641, 1875.682, 1918.298, 1961.330, 2005.110, 2046.810],
    'NAND':     [7518.504, 7713.473, 7914.411, 8102.195, 8176.052, 8300.919, 8490.030, 8677.650, 8857.050, 9030.220],
    'NOR':      [8200.430, 8496.150, 8785.500, 9082.110, 9192.350, 9384.430, 9662.640, 9965.700, 10246.240, 10517.620],
}

# --- 4. AVERAGE DYNAMIC POWER (pW) ---
dynamic_avg = {
    'Inverter': [1709.57945, 1747.50350, 1787.25400, 1831.70150, 1843.64100, 1877.54250, 1920.32400, 1963.37700, 2006.81500, 2047.46000],
    'NAND':     [7520.76125, 7715.62625, 7916.29375, 8103.51225, 8177.06225, 8301.87150, 8491.69975, 8680.25750, 8863.19250, 9044.94750],
    'NOR':      [8208.373075, 8504.254225, 8793.964175, 9091.290525, 9201.897000, 9394.840250, 9674.975250, 9980.760500, 10265.000000, 10540.397500],
}

# Build one combined DataFrame for easy viewing
df = pd.DataFrame({
    'Temperature': temps,
    'TotalAvg_Inverter': total_avg['Inverter'],
    'TotalAvg_NAND':     total_avg['NAND'],
    'TotalAvg_NOR':      total_avg['NOR'],
    'Static_Inverter':   static_power['Inverter'],
    'Static_NAND':       static_power['NAND'],
    'Static_NOR':        static_power['NOR'],
    'DynWorst_Inverter': dynamic_worst['Inverter'],
    'DynWorst_NAND':     dynamic_worst['NAND'],
    'DynWorst_NOR':      dynamic_worst['NOR'],
    'DynAvg_Inverter':   dynamic_avg['Inverter'],
    'DynAvg_NAND':       dynamic_avg['NAND'],
    'DynAvg_NOR':        dynamic_avg['NOR'],
})

print('=' * 80)
print('YOUR COMPLETE RESEARCH DATASET — 4 Power Types × 3 Gates')
print('=' * 80)
print(df.to_string(index=False))
print(f'\nShape: {df.shape[0]} temperatures × {df.shape[1]} columns (1 temp + 12 power values)')

## 📈 Step 2: Visualize All 4 Power Types

In [ ]:
T = temps
colors = {'Inverter': '#2196F3', 'NAND': '#4CAF50', 'NOR': '#FF5722'}
markers = {'Inverter': 'o', 'NAND': 's', 'NOR': '^'}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Temperature vs Power — All 4 Power Types\n(Your Research Data)', fontsize=15, fontweight='bold')

datasets = [
    (axes[0,0], total_avg,     'Total Average Power (pW)'),
    (axes[0,1], static_power,  'Static Power — Worst Case (pW)'),
    (axes[1,0], dynamic_worst, 'Dynamic Power — Worst Case (pW)'),
    (axes[1,1], dynamic_avg,   'Average Dynamic Power (pW)'),
]

for ax, data, title in datasets:
    for gate in ['Inverter', 'NAND', 'NOR']:
        ax.plot(T, data[gate], marker=markers[gate], color=colors[gate],
                linewidth=2, markersize=7, label=gate,
                markerfacecolor='white', markeredgewidth=2)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Temperature (°C)')
    ax.set_ylabel('Power (pW)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_facecolor('#f8f9fa')

plt.tight_layout()
plt.savefig('all_power_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print('💡 OBSERVATIONS:')
print('  • Total Avg Power    → near-linear increase with temperature')
print('  • Static Power       → exponential increase (leakage doubles every ~20°C)')
print('  • Dynamic Worst Case → near-linear (dominates total power)')
print('  • Avg Dynamic Power  → near-linear, slightly less than worst case')
print('  • NOR gate consumes more power than NAND, which is more than Inverter')

## 🤖 Step 3: Train ML Models for ALL 12 Power Targets
### (4 power types × 3 gates = 12 models total)

In [ ]:
X = np.array(temps)

def train_and_evaluate(X, y, target_name):
    X_r = X.reshape(-1, 1)
    loo = LeaveOneOut()

    # Model 1: Linear Regression
    lin = LinearRegression()
    lin_cv = cross_val_score(lin, X_r, y, cv=loo, scoring='r2').mean()
    lin.fit(X_r, y)

    # Model 2: Polynomial Regression (degree 2)
    poly = Pipeline([('poly', PolynomialFeatures(degree=2, include_bias=False)),
                     ('lin',  LinearRegression())])
    poly_cv = cross_val_score(poly, X_r, y, cv=loo, scoring='r2').mean()
    poly.fit(X_r, y)

    # Model 3: Random Forest
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_cv = cross_val_score(rf, X_r, y, cv=loo, scoring='r2').mean()
    rf.fit(X_r, y)

    models = {
        'Linear':     {'model': lin,  'cv_r2': lin_cv,  'r2': r2_score(y, lin.predict(X_r)),  'mae': mean_absolute_error(y, lin.predict(X_r))},
        'Polynomial': {'model': poly, 'cv_r2': poly_cv, 'r2': r2_score(y, poly.predict(X_r)), 'mae': mean_absolute_error(y, poly.predict(X_r))},
        'RandomForest':{'model': rf,  'cv_r2': rf_cv,   'r2': r2_score(y, rf.predict(X_r)),   'mae': mean_absolute_error(y, rf.predict(X_r))},
    }
    best = max(models, key=lambda k: models[k]['cv_r2'])
    return models, best


# ================================================================
# TRAIN ALL 12 MODELS
# ================================================================
all_results = {}

power_groups = [
    ('Total Avg Power',          total_avg,     'TotalAvg'),
    ('Static Power (worst)',     static_power,  'Static'),
    ('Dynamic Power (worst)',    dynamic_worst, 'DynWorst'),
    ('Average Dynamic Power',    dynamic_avg,   'DynAvg'),
]

print(f'{"="*80}')
print(f'  {"Target":<35} {"Best Model":<14} {"R² (train)":>10} {"CV R²":>8} {"MAE (pW)":>10}')
print(f'  {"-"*80}')

for group_name, data_dict, key_prefix in power_groups:
    print(f'\n  [{group_name}]')
    for gate in ['Inverter', 'NAND', 'NOR']:
        y = np.array(data_dict[gate])
        key = f'{key_prefix}_{gate}'
        results, best = train_and_evaluate(X, y, key)
        all_results[key] = {'results': results, 'best': best, 'y': y}
        r2  = results[best]['r2']
        cv  = results[best]['cv_r2']
        mae = results[best]['mae']
        print(f'  {group_name} — {gate:<10} {best:<14} {r2:>10.6f} {cv:>8.4f} {mae:>10.4f}')

print(f'\n{"="*80}')
print('✅ All 12 models trained successfully!')

## 📊 Step 4: Plot Predictions vs Actual — All 4 Power Types

In [ ]:
T_fine = np.linspace(-40, 120, 300).reshape(-1, 1)
X_r    = X.reshape(-1, 1)
model_colors = {'Linear': '#e74c3c', 'Polynomial': '#2ecc71', 'RandomForest': '#3498db'}

fig, axes = plt.subplots(4, 3, figsize=(20, 22))
fig.suptitle('ML Model Predictions vs Actual Data — All 4 Power Types × 3 Gates',
             fontsize=14, fontweight='bold')

plot_rows = [
    (total_avg,     'TotalAvg',  'Total Avg Power'),
    (static_power,  'Static',    'Static Power (worst)'),
    (dynamic_worst, 'DynWorst',  'Dynamic Power (worst)'),
    (dynamic_avg,   'DynAvg',    'Avg Dynamic Power'),
]

for row_idx, (data_dict, key_prefix, row_label) in enumerate(plot_rows):
    for col_idx, gate in enumerate(['Inverter', 'NAND', 'NOR']):
        ax = axes[row_idx][col_idx]
        key = f'{key_prefix}_{gate}'
        y_actual = np.array(data_dict[gate])
        res = all_results[key]['results']

        ax.scatter(X, y_actual, color='black', s=60, zorder=5,
                   label='Actual', marker='D')

        for mname, mcolor in model_colors.items():
            pred = res[mname]['model'].predict(T_fine)
            r2   = res[mname]['r2']
            ax.plot(T_fine, pred, color=mcolor, linewidth=1.8,
                    label=f'{mname} R²={r2:.4f}')

        ax.set_title(f'{row_label}\n{gate}', fontweight='bold', fontsize=9)
        ax.set_xlabel('Temperature (°C)', fontsize=8)
        ax.set_ylabel('Power (pW)', fontsize=8)
        ax.legend(fontsize=6, loc='upper left')
        ax.grid(True, alpha=0.3)
        ax.set_facecolor('#f8f9fa')

plt.tight_layout()
plt.savefig('all_model_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved as all_model_predictions.png')

## 🎯 Step 5: Predict at Any Temperature — All 4 Power Types

In [ ]:
def predict_all(temperature_C):
    T_in = np.array([[temperature_C]])
    print(f'\n🌡️  PREDICTIONS FOR TEMPERATURE = {temperature_C}°C')
    print('=' * 80)
    print(f'  {"Power Type":<30} {"Gate":<12} {"Linear":>12} {"Polynomial":>12} {"RandomForest":>14}')
    print(f'  {"-"*82}')

    for group_name, data_dict, key_prefix in [
        ('Total Avg Power',       total_avg,     'TotalAvg'),
        ('Static Power (worst)',  static_power,  'Static'),
        ('Dynamic Power (worst)', dynamic_worst, 'DynWorst'),
        ('Avg Dynamic Power',     dynamic_avg,   'DynAvg'),
    ]:
        for gate in ['Inverter', 'NAND', 'NOR']:
            key = f'{key_prefix}_{gate}'
            res = all_results[key]['results']
            lin_p  = res['Linear']['model'].predict(T_in)[0]
            poly_p = res['Polynomial']['model'].predict(T_in)[0]
            rf_p   = res['RandomForest']['model'].predict(T_in)[0]
            print(f'  {group_name:<30} {gate:<12} {lin_p:>12.3f} {poly_p:>12.3f} {rf_p:>14.3f}')
        print()


# ==========================================
# CHANGE THESE TEMPERATURES AS YOU NEED!
# ==========================================
predict_all(50)    # 50°C
predict_all(85)    # 85°C
predict_all(110)   # 110°C

## 📐 Step 6: Polynomial Equations for Research Paper

In [ ]:
print('📐 POLYNOMIAL EQUATIONS (Degree 2) — Power = a·T² + b·T + c')
print('   Copy these directly into your research paper!')
print('=' * 80)

for group_name, data_dict, key_prefix in [
    ('TOTAL AVERAGE POWER',       total_avg,     'TotalAvg'),
    ('STATIC POWER (WORST CASE)', static_power,  'Static'),
    ('DYNAMIC POWER (WORST CASE)',dynamic_worst, 'DynWorst'),
    ('AVERAGE DYNAMIC POWER',     dynamic_avg,   'DynAvg'),
]:
    print(f'\n  [{group_name}]')
    for gate in ['Inverter', 'NAND', 'NOR']:
        key   = f'{key_prefix}_{gate}'
        model = all_results[key]['results']['Polynomial']['model']
        coef  = model.named_steps['lin'].coef_
        inter = model.named_steps['lin'].intercept_
        a, b, c = coef[1], coef[0], inter
        r2 = all_results[key]['results']['Polynomial']['r2']
        print(f'  {gate:<10}: Power = {a:.8f}·T² + {b:.6f}·T + {c:.4f}  (R²={r2:.8f})')

print('\n  T = Temperature in °C | Power in pW')

## 💾 Step 7: Save All 12 Models

In [ ]:
os.makedirs('saved_models', exist_ok=True)

for key, val in all_results.items():
    best_model = val['results'][val['best']]['model']
    path = f'saved_models/{key}_best_model.pkl'
    joblib.dump(best_model, path)
    print(f'✅ Saved: {path}  (Best: {val["best"]})')

print(f'\n🎉 All 12 models saved in saved_models/ folder!')
print('\nTo reload later:')
print("""
    import joblib, numpy as np
    model = joblib.load('saved_models/DynWorst_Inverter_best_model.pkl')
    pred  = model.predict([[temperature_in_celsius]])
    print(f'Dynamic Power (Inverter): {pred[0]:.3f} pW')
""")

## 📊 Step 8: Final Summary Table for Research Paper
> Copy this table directly into your paper!

In [ ]:
rows = []
for group_name, key_prefix in [
    ('Total Avg Power',       'TotalAvg'),
    ('Static Power (worst)',  'Static'),
    ('Dynamic Power (worst)', 'DynWorst'),
    ('Avg Dynamic Power',     'DynAvg'),
]:
    for gate in ['Inverter', 'NAND', 'NOR']:
        key  = f'{key_prefix}_{gate}'
        best = all_results[key]['best']
        r2   = all_results[key]['results'][best]['r2']
        cv   = all_results[key]['results'][best]['cv_r2']
        mae  = all_results[key]['results'][best]['mae']
        rows.append({'Power Type': group_name, 'Gate': gate,
                     'Best Model': best, 'R²': round(r2, 6),
                     'CV R²': round(cv, 4), 'MAE (pW)': round(mae, 4)})

summary = pd.DataFrame(rows)
print('📋 COMPLETE MODEL PERFORMANCE SUMMARY')
print('=' * 80)
print(summary.to_string(index=False))
summary.to_csv('model_summary.csv', index=False)
print('\n✅ Summary saved as model_summary.csv')

## 🎓 You're Done!

**Files created:**
- `all_power_visualization.png` — plots of all 4 power types
- `all_model_predictions.png` — model fit plots (12 subplots)
- `model_summary.csv` — complete R², MAE table for your paper
- `saved_models/` — folder with all 12 trained models

**Power types covered:**
- ✅ Total Average Power
- ✅ Static Power (worst case)
- ✅ Dynamic Power (worst case)
- ✅ Average Dynamic Power